In [3]:
import numpy as np
import rasterio as rio
from rasterio.enums import Resampling

In [10]:
def upscale_raster(input_raster, output_raster, scale_factor=10):
    """
    Reads a high-resolution (2 cm) raster and upscales it to a lower resolution (10 cm)
    by averaging the pixel values within each new upscaled pixel.

    Parameters:
    - input_raster (str): Path to the input raster file.
    - output_raster (str): Path to save the upscaled raster.
    - scale_factor (int): The factor by which to upscale (e.g., 2cm → 10cm = 5x scaling).
    """
    with rio.open(input_raster) as src:
        # Read the data as a NumPy array
        data = src.read()  # Read first band
        profile = src.profile

        # Get original dimensions
        bands, height, width = data.shape
        new_height, new_width = height // scale_factor, width // scale_factor

        # Reshape and average in blocks of scale_factor x scale_factor
        reshaped_data = data[:, :new_height * scale_factor, :new_width * scale_factor].reshape(
            bands, new_height, scale_factor, new_width, scale_factor)
        upscaled_data = reshaped_data.mean(axis=(2, 4))  # Average over 2 dimensions

        # Update metadata for new resolution
        profile.update(
            dtype=rio.float32,
            height=new_height,
            width=new_width,
            transform=rio.transform.Affine(
                src.transform.a * scale_factor, src.transform.b, src.transform.c,
                src.transform.d, src.transform.e * scale_factor, src.transform.f
            ),
            count=bands
        )

        # Write the upscaled raster
        with rio.open(output_raster, "w", **profile) as dst:
            dst.write(upscaled_data.astype(rio.float32))

    return None

In [11]:
# HiLDEN: TOR_KOMR_20180805
upscale_raster("/projectnb/modislc/users/seamorez/HLS_FCover/UAV/TOR/CA_TOR_KOMR_RGB_20180805/CA_TOR_KOMR_RGB_20180805_rgb.tif", 
               "/projectnb/modislc/users/seamorez/HLS_FCover/UAV/TOR/CA_TOR_KOMR_RGB_20180805/CA_TOR_KOMR_RGB_20180805_rgb_20cm.tif")